### Normalization consists of the following line-by-line process:

~~1. Remove primarily non-English songs~~

~~4. Add relative start and end times~~

~~5. Remove chunks longer than 10s~~
    - ~~This is less than 1% of chunks~~

~~2. Change British spellings to American
    - This should be done initially AND after misspellings are taken care of (misspellings could reintroduce British, which must be eliminated). But also, would two words ever be joined into a British word? vig our -> vigour?~~

~~3. Remove all punctuation except contraction/possessive/slang apostrophes like can't/john's/'cause~~

2. Combine incorrectly split words and separate invalid compound words
    - Currently, performed before misspellings.
    - Some words, like hmm, moo, bzz need two consecutive letters
    - Most words, like fuuck, do not
    - So after reducing 3+ occurrences to 2, is a dictionary/wordfreq check good enough to say whether an additional letter should be deleted?
    - Most legit way to do this would be to check how such words are tokenized in Whisper model
    - woo, oh, no, ah, eyow, go, hm | hmm, 
    - what about go-o vs goo?
    - leave lalala alone
    - change yey to yay
    - Might be better to use known dictionary on first pass to get dictionary-standard words before dealing with slang/spelling variants
    - Backup word test could be passes zipf_frequency test
    - If not, check if it should be combined with a nearby word fragment(s) to create an actual word, e.g. ci ty -> city. Or a double letter should be changed to a single letter, e.g. "yees" -> "yes"
    - Deal with misspellings by combining spell-checker and phonetics-checker

3. Deal with misspellings

    - Going assumption is that there won't be many misspellings. Will check misspellings after British->American and splitting/combining words.

3. (Later) Use transformers to detect new slang (and don't consider new slang a misspelling)

~~3. Change numbers like 21 into text, so twenty-one~~




In [2]:
import pandas as pd
import ast
from wordfreq import zipf_frequency
import jiwer
import re
import numpy as np
import enchant
from spellchecker import SpellChecker
from itertools import combinations
from transformers import WhisperProcessor
import fuzzy
from num2words import num2words
import inflect
import os
import csv
import soundfile as sf
from functools import lru_cache

In [3]:
# First and last names used in checking if lyrics are valid words
df_male = pd.read_csv('./data/male.txt',header=None,names=["name"])
df_female = pd.read_csv('./data/female.txt',header=None,names=["name"])
df_last = pd.read_csv('./data/Names_2010Census.csv',usecols = ['name'])

df_names = pd.concat([df_male, df_female,df_last], ignore_index=True)

# British spellings used to correct spelling later
brit_to_us = pd.read_json('./data/british_to_american_sp.json', orient='index')
brit_to_us.reset_index(inplace=True)
brit_to_us.columns = ['british', 'american']
brit_to_us = brit_to_us[~brit_to_us['british'].isin(['aeroplane', 'aluminium', 'buses', 'axe', 'aesthetic'])]

In [4]:
# Clean metadata-full-lines.csv to have same format as metadata-lines.csv
df = pd.read_csv("../../data/metadata-full-lines.csv", usecols=["filename", "words","transcript","start","end"])
df = df.rename(columns={"words": "transcript","transcript":"words", "start":"abs_start", "end":"abs_end"})
df = df[["filename", "transcript","abs_start","abs_end","words"]]

In [5]:
# Remove bad songs (non-English, transcription/lyrics issues, etc.) by reading in and cleaning in bad_songs.csv
df_bad = pd.read_csv('./data/bad_songs.csv')
df_bad['filename'] = df_bad['filename'].str.replace(r'^\d+\.\s*', '', regex=True)   # remove number + period + spaces at the start
df_bad['filename'] = df_bad['filename'].str.replace(r'\s+', '', regex=True) # delete whitespace

# Build a tuple of bad prefixes
bad_prefixes = tuple(df_bad['filename'].values)

# Filter rows where 'filename' starts with any bad prefix
mask = df['filename'].str.startswith(bad_prefixes)
df = df[~mask]

In [6]:
# Add new columns for relative start and end
df['rel_start']=0.0
df['rel_end']=df['abs_end']-df['abs_start']

# Discard rows longer than 10s
df = df[df['rel_end']<=10.0]

# Discard rows shorter than 0.5s
df = df[df['rel_end']>=1]

In [7]:
# Next two functions createa a .csv from audio chunks with Is_Silent column signaling either silence or extremely low audio

def analyze_audio(file_path, threshold=0.01):
    """
    Analyze a single audio file.
    Returns: (peak, rms, is_silent)
    """
    try:
        data, sr = sf.read(file_path)

        # Handle empty files
        if data.size == 0:
            return 0.0, 0.0, True  # Consider empty files as silent

        # Convert stereo to mono if needed
        if data.ndim > 1:
            data = np.mean(data, axis=1)

        peak = float(np.max(np.abs(data)))
        rms = float(np.sqrt(np.mean(data ** 2)))
        is_silent = peak < threshold
        return peak, rms, is_silent

    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None, None, None


def process_folder_with_df(folder_path, df, output_csv="audio_analysis.csv", threshold=0.01):
    """
    Analyze .wav files in a folder, print matching transcript and words from df,
    and save results to CSV.
    """
    results = []

    for fname in os.listdir(folder_path):
        if fname.lower().endswith(".wav"):
            path = os.path.join(folder_path, fname)
            peak, rms, silent = analyze_audio(path, threshold)

            if peak is not None:
                # Find matching row in df
                row = df[df['filename'] == fname]
                if not row.empty:
                    transcript = row.iloc[0]['transcript_no_punct']
                    words = row.iloc[0]['words']
                else:
                    transcript = "[No transcript found]"
                    words = "[No words found]"

                results.append([fname, peak, rms, silent, transcript, words])

    # Save results to CSV
    csv_path = os.path.join("./data/", output_csv)
    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Filename", "Peak", "RMS", "Is_Silent", "Transcript", "Words"])
        writer.writerows(results)

    print(f"\nAnalysis complete. Results saved to {csv_path}")


# Apply previous two functions to remove silence and create .csv
# Uncomment these lines out to update .csv
# Takes my computer ~25min to run
#folder = "../../data/DALI-chunks-lines/chunked/"
#process_folder_with_df(folder, df, output_csv="audio_analysis_with_text.csv", threshold=0.03)

# Read in the CSV
df_analysis = pd.read_csv("./data/audio_analysis_with_text.csv")

# Count how many rows have Is_Silent = True
silent_count = df_analysis[df_analysis['Is_Silent'] == True].shape[0]

# Merge the Is_Silent column into df_main based on matching filenames
try:
    df['Is_Silent']

except:
    df = df.merge(
        df_analysis[['Filename', 'Is_Silent']],
        how='left',
        left_on='filename',   # column in df_main
        right_on='Filename'   # column in df_analysis
    )

    # Drop duplicate 'Filename' column if needed
    df.drop(columns=['Filename'], inplace=True)

    # Drop rows where Is_Silent is True
    df = df[df['Is_Silent'] == False].reset_index(drop=True)

In [8]:
# Clean text: delete puctuation (keep apostrophes), collapse multiple spaces
df['transcript_no_punct'] = (
    df['transcript']
    .str.replace(r'(?<=\S)-(?=\S)', ' ', regex=True)  # Hyphen surrounded by non-space
    .str.replace(r'\s*-\s*', '', regex=True) # Hyphen with space on either side
    .str.replace(r"[^\w\s'-]", '', regex=True)  # Remove unwanted punctuation
    .str.replace(r"_", "", regex=True)          # Remove underscores
    .str.replace(r"\s+", ' ', regex=True)       # Collapse multiple spaces
    .str.replace(r"^'+(.*?)'+$", r"\1", regex=True) # Happens twice to remove outer apostrophes for things like ''you are''
    .str.replace(r"^'+(.*?)'+$", r"\1", regex=True) # See above
    .str.strip()
)

In [ ]:
# Below is a collection of functions to further clean the dataset later

# Check if a string contains the same character 3 or more times in a row
def three_or_more_repeats(text):
    return bool(re.search(r"(.)\1{2,}", text))

# Collapse 3+ letters to 2
def collapse_repeats(text):
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

# Further collapse certain words from 1 repeat to 0 repeats
def collapse_known_repeats(text):
    known_patterns = {"noo", "ohh", "yess", "goo", "aah", "woahh", "laa", "poww", "hii", "heyy", "ayy", "okayy", "byee", "yeahh", "sayy"}    
    
    def collapse(word):
            if word in known_patterns:
                # Collapse all double letters in the word down to one occurrence
                word = re.sub(r'(.)\1+', r'\1', word)
            return word

    return ' '.join(collapse(word) for word in text.split())

# Convert 1- and 2-digit numbers to text, e.g. 21 -> twenty-one
# Convert 3 digit numbers based on common pronunciations, e.g. 247 -> twenty-four seven, 101 -> one-oh-one
from num2words import num2words
import inflect
import re

p = inflect.engine()

def convert_numbers_to_words(text):
    def ordinal_replacer(match):
        number = match.group(1)
        try:
            return num2words(int(number), to='ordinal')
        except:
            return match.group(0)

    def plural_decade_replacer(match):
        number = match.group(1)
        try:
            base = num2words(int(number))
            return p.plural(base)
        except:
            return match.group(0)

    def number_replacer(match):
        raw = match.group(0)
        number = raw.lstrip("'")

        try:
            if number == "0":
                return "oh"
            if number == "34":
                return "three four"
            if number == "247":
                return "twenty four seven"
            if number == "730":
                return "seven thirty"
            if number == "1000":
                return "thousand"
            if number == "1954":
                return "nineteen fifty four"
            if number == "130":
                return "one thirty"
            if int(number) > 99 and int(number)<1000:
                return ' '.join('oh' if d == '0' else num2words(int(d)) for d in number)
            else:
                words = num2words(int(number))  # cardinal default
                return words.replace("-", " ")  # remove hyphens
        except:
            return raw

    # 1. Convert 8th → eighth
    text = re.sub(r"\b(\d+)th\b", ordinal_replacer, text)

    # 2. Convert 40's, 30s → forties, thirties
    text = re.sub(r"\b(\d{2})'?s\b", plural_decade_replacer, text)

    # 3. Convert regular numbers (with optional apostrophe prefix)
    text = re.sub(r"'?\d+", number_replacer, text)

    return text



def normalize_expressive_words(text):
    # Replace "whoo" → "woo"
    text = re.sub(r'\bwhoo\b', 'woo', text, flags=re.IGNORECASE)
    # Replace "yey" → "yay"
    text = re.sub(r'\byey\b', 'yay', text, flags=re.IGNORECASE)
    return text



# Spell-checker used to check for misspellings. First, go through known words, then names. If a word is neither, 
d = enchant.Dict("en_US")
spell = SpellChecker()

def is_dictionary_word(word):
    if word in {'a','i'}:
        return True
    if len(word)>1:
        if d.check(word):
            return True
        # spell allows for "words" like "ni", "th", etc...
        #if word in spell:
            #return True
    return False

def is_name(word):
    if (df_names['name'].str.lower() == word).any():
        #print('is a name')
        return True
    return False

def is_dictionary_word_or_name(word):
    if is_dictionary_word(word) or is_name(word):
        return True
    no_g_word = word[:-1] + 'g' # checkin' <-> checking
    if is_dictionary_word(no_g_word) or is_name(no_g_word):
        return True
    no_apo_word = word[1:]  # 'cause <-> cause
    if is_dictionary_word(no_apo_word) or is_name(no_apo_word):
        return True
    return False

# Used for output formatting
def add_quotes(word):
    return f'"{word}"'


def join_non_words(text):
    words = text.split()
    result = []
    i = 0
    while i < len(words):
        word = words[i]

        # Case 1: current word is valid -> keep
        if is_dictionary_word_or_name(word):
            result.append(word)
            i += 1

        # Case 2: Try forward join with next word
        elif i + 1 < len(words):
            forward = word + words[i + 1]
            if is_dictionary_word_or_name(forward):
                result.append(forward)
                i += 2
            else:
                # Case 3: Try backward join with previous word
                if result:
                    backward = result[-1] + word
                    if is_dictionary_word_or_name(backward):
                        result.pop()              # remove previous
                        result.append(backward)   # add merged
                        i += 1
                        continue
                result.append(word)
                i += 1

        # Case 4: Last word, no forward option
        else:
            if result:
                backward = result[-1] + word
                if is_dictionary_word_or_name(backward):
                    result.pop()
                    result.append(backward)
                else:
                    result.append(word)
            else:
                result.append(word)
            i += 1

    return ' '.join(result)


# Correct misspellings based on spell checker and phonetics checker
dmeta = fuzzy.DMetaphone()

def correct_misspelling(misspelled_word):
    goal_phonetics = dmeta(misspelled_word)
    matches = []
    if is_dictionary_word(misspelled_word) or is_name(misspelled_word):
        return misspelled_word
    candidates = spell.candidates(misspelled_word)
    if candidates is not None:
        for word in candidates:
            if dmeta(word) == goal_phonetics:
                matches.append(word)
    if len(matches) == 0:
        print(misspelled_word,"is likely misspelled, but couldn't find a match!")
        return misspelled_word
    if len(matches) == 1:
        if word in matches:
            print(word,'is already a word...')
            return matches[0]
        print("Found one match:",add_quotes(misspelled_word),"corrects to",add_quotes(matches[0]))
        return matches[0]
    if len(matches) > 1:
        print("Found more than one match for",add_quotes(misspelled_word),"so no action taken.")
        return misspelled_word


# For a given nonword, break it into 2+ pieces to see if those pieces are words
def split_and_check(word):
    length = len(word)

    for num_pieces in range(2, 6): # range(2, 4)
        # Generate all possible split positions for num_pieces
        for split_points in combinations(range(1, length), num_pieces - 1):
            indices = (0,) + split_points + (length,)
            pieces = [word[indices[i]:indices[i + 1]] for i in range(len(indices) - 1)]

            if all(is_dictionary_word(piece) for piece in pieces):
                if num_pieces >= 4:
                    print('found a',num_pieces, 'parter:',word,pieces)
                return pieces

    return False

# Remove British text
brit_to_us_dict = dict(zip(brit_to_us['british'], brit_to_us['american']))

def convert_british_to_american(text):
    words = text.split()
    converted = [
        brit_to_us_dict.get(word, word)  # Replace if found, else keep word
        for word in words
    ]
    return ' '.join(converted)


# Define fragments (longest first so regex prefers the longer match when overlapping)
_EXPRESSIVE_FRAGS = ("woah", "doo", "la", "ooh", "doh", "da", "oh", "dub", "du")    # order matters in this list
_EXPRESSIVE_RE = re.compile(
    "|".join(sorted(_EXPRESSIVE_FRAGS, key=len, reverse=True)),
    re.IGNORECASE
)


def split_expressive_word(word, pattern=_EXPRESSIVE_RE, min_frags=2, canonical_lower=False):
    matches = list(pattern.finditer(word))
    if len(matches) < min_frags:
        return [word]

    parts = []
    last = 0
    for m in matches:
        if m.start() > last:
            parts.append(word[last:m.start()])
        frag = m.group(0)
        parts.append(frag.lower() if canonical_lower else frag)
        last = m.end()

    if last < len(word):
        parts.append(word[last:])

    # Drop any empty strings that can arise if matches are abutting
    return [p for p in parts if p]


def split_expressive_string(text, pattern=_EXPRESSIVE_RE, min_frags=2, canonical_lower=False):
    out_tokens = []
    for w in text.split():
        out_tokens.extend(
            split_expressive_word(
                w,
                pattern=pattern,
                min_frags=min_frags,
                canonical_lower=canonical_lower,
            )
        )
    return " ".join(out_tokens)

In [46]:
FRAGMENTS = ("woah", "whoa", "who", "wo", "go", "doo", "la", "uoh", "ooh", "ohh", "doh", "dab", "da", "oh", "dum", "dub", "du", "sha", "eh")
FRAG_RE = re.compile("|".join(sorted(FRAGMENTS, key=len, reverse=True)), re.IGNORECASE)

# Words we will never split
EXCLUDED_WORDS = {"mandalay", "modular", "interactiveodular", "undulates", "enchilada", "lagoon", "shalaby"}

def split_expressive_word_adjacent(word, fragments=FRAGMENTS, min_frags=2, canonical_lower=False):
    # Skip excluded words
    if word.lower() in EXCLUDED_WORDS:
        return [word]

    # Find all fragments in the word
    matches = list(FRAG_RE.finditer(word))
    if not matches:
        return [word]
    
    # Check if there are at least 2 adjacent fragments
    positions = [m.span() for m in matches]
    adjacent = 0
    last_end = None
    for start, end in positions:
        if last_end == start:  # directly adjacent
            adjacent += 1
        last_end = end
    if adjacent == 0:
        return [word]

    # Split the word
    parts, last = [], 0
    for m in matches:
        if m.start() > last:
            parts.append(word[last:m.start()])
        frag = m.group(0)
        parts.append(frag.lower() if canonical_lower else frag)
        last = m.end()
    if last < len(word):
        parts.append(word[last:])
    return [p for p in parts if p]

def split_expressive_string_adjacent(text):
    out = []
    for w in text.split():
        out.extend(split_expressive_word_adjacent(w))
    return " ".join(out)

# Fix words du -> do, woah -> whoa
#def normalize_variants(text):
    # Use regex with \b to match word boundaries
    #return re.sub(r'\bdu{1,2}\b', 'do', text, flags=re.IGNORECASE)

# Fix words: a way -> away, per son -> person
def fix_common_splits(text: str) -> str:
    # Use word boundaries to avoid partial matches
    #text = re.sub(r'\ba way\b', 'away', text, flags=re.IGNORECASE)
    text = re.sub(r'\bdu{1,2}\b', 'do', text, flags=re.IGNORECASE)
    text = re.sub(r'\ba gain\b', 'again', text, flags=re.IGNORECASE)
    text = re.sub(r'\bper son\b', 'person', text, flags=re.IGNORECASE)
    text = re.sub(r'\bgot ta\b', 'gotta', text, flags=re.IGNORECASE)
    text = re.sub(r'\bin side\b', 'inside', text, flags=re.IGNORECASE)
    text = re.sub(r'\byour self\b', 'yourself', text, flags=re.IGNORECASE)
    text = re.sub(r'\bno thing\b', 'nothing', text, flags=re.IGNORECASE)
    text = re.sub(r'\bholly wood\b', 'hollywood', text, flags=re.IGNORECASE)
    text = re.sub(r'\bye ah\b', 'yeah', text, flags=re.IGNORECASE)
    text = re.sub(r'\byea ah\b', 'yeah', text, flags=re.IGNORECASE)
    text = re.sub(r'\bsome where\b', 'somewhere', text, flags=re.IGNORECASE)
    text = re.sub(r'\ba lone\b', 'alone', text, flags=re.IGNORECASE)
    text = re.sub(r'\bii\b', 'i', text, flags=re.IGNORECASE)
    text = re.sub(r'\bsome one\b', 'someone', text, flags=re.IGNORECASE)
    text = re.sub(r'\bi m\b', 'i\'m', text, flags=re.IGNORECASE)
    text = re.sub(r'\bgoo\b', 'go', text, flags=re.IGNORECASE)
    text = re.sub(r'\btho ugh\b', 'though', text, flags=re.IGNORECASE)
    text = re.sub(r'\bmid night\b', 'midnight', text, flags=re.IGNORECASE)
    text = re.sub(r'\bthrough out\b', 'throughout', text, flags=re.IGNORECASE)
    text = re.sub(r'\bsome times\b', 'sometimes', text, flags=re.IGNORECASE)
    text = re.sub(r'\brain bow\b', 'rainbow', text, flags=re.IGNORECASE)
    text = re.sub(r'\with out\b', 'without', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwant ed\b', 'wanted', text, flags=re.IGNORECASE)
    text = re.sub(r'\bi ve\b', 'i\'ve', text, flags=re.IGNORECASE)
    text = re.sub(r'\bhes\b', 'he\'s', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhats\b', 'what\'s', text, flags=re.IGNORECASE)
    text = re.sub(r'\btheres\b', 'there\'s', text, flags=re.IGNORECASE)
    text = re.sub(r'\bthats\b', 'that\'s', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhos\b', 'who\'s', text, flags=re.IGNORECASE)
    text = re.sub(r'\bhows\b', 'how\'s', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhens\b', 'when\'s', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhys\b', 'why\'s', text, flags=re.IGNORECASE)
    text = re.sub(r'\bdont\b', 'don\'t', text, flags=re.IGNORECASE)
    text = re.sub(r'\bdont\b', 'isn\'t', text, flags=re.IGNORECASE)
    text = re.sub(r'\baint\b', 'ain\'t', text, flags=re.IGNORECASE)
    text = re.sub(r'\btheyd\b', 'they\'d', text, flags=re.IGNORECASE)
    text = re.sub(r'\bbrazillian\b', 'brazilian', text, flags=re.IGNORECASE)
    text = re.sub(r'defence', 'defense', text, flags=re.IGNORECASE)
    text = re.sub(r'\bhappyness\b', 'happiness', text, flags=re.IGNORECASE)
    text = re.sub(r'\bcrazyness\b', 'craziness', text, flags=re.IGNORECASE)
    text = re.sub(r'\ballways\b', 'always', text, flags=re.IGNORECASE)
    text = re.sub(r'\byoull\b', 'you\'ll', text, flags=re.IGNORECASE)
    text = re.sub(r'\bcan not\b', 'cannot', text, flags=re.IGNORECASE)
    text = re.sub(r'\bjudgement\b', 'judgment', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhereever\b', 'wherever', text, flags=re.IGNORECASE)
    text = re.sub(r'\ban other\'s\b', 'another\'s', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhoah\b', 'whoa', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhoahh\b', 'whoa', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhoaah\b', 'whoa', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhooah\b', 'whoa', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhooaah\b', 'whoa', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhooahh\b', 'whoa', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhooaahh\b', 'whoa', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhoaahh\b', 'whoa', text, flags=re.IGNORECASE)
    text = re.sub(r'\bcadilac\b', 'cadillac', text, flags=re.IGNORECASE)
    text = re.sub(r'\bdevine\b', 'divine', text, flags=re.IGNORECASE)
    text = re.sub(r'\bwhen ever\b', 'whenever', text, flags=re.IGNORECASE)
    text = re.sub(r'\bbyrds\b', 'birds', text, flags=re.IGNORECASE)
    text = re.sub(r'\bdeply\b', 'deeply', text, flags=re.IGNORECASE)
    text = re.sub(r'\bengrained\b', 'ingrained', text, flags=re.IGNORECASE)
    text = re.sub(r'\bslained\b', 'slain', text, flags=re.IGNORECASE)
    text = re.sub(r'\blullabye\b', 'lullaby', text, flags=re.IGNORECASE)



    # substring instead of subword
    text = re.sub(r'defence', 'defense', text, flags=re.IGNORECASE)
    return text



In [ ]:
# ------------------------------------------------------------------
# 1. REQUIRED INPUTS YOU PROVIDE
# ------------------------------------------------------------------
# d: a spell/dictionary object with .check(str) -> bool
# df_names: DataFrame with a 'name' column (strings)
#
# Example (uncomment / adapt as needed):
# import enchant
# d = enchant.Dict("en_US")
# df_names = pd.DataFrame({"name": ["Alice", "Bob", "Dallas", "Jared"]})

# NOTE: The code below expects d and df_names to exist in the namespace.
# ------------------------------------------------------------------


# ------------------------------------------------------------------
# 2. BUILD NAME LOOKUP SET (FAST)
# ------------------------------------------------------------------
def build_names_set(df_names, col="name"):
    """Precompute lowercase name set for O(1) membership."""
    return {str(x).strip().lower() for x in df_names[col].dropna()}


# ------------------------------------------------------------------
# 3. CONFIGURE "NOT WORDS" FORCE-EXCLUSION SET
#    (Extend this at will; lowercase only)
# ------------------------------------------------------------------
NOT_WORDS = {}


# ------------------------------------------------------------------
# 4. CORE LOOKUP HELPERS (CACHED)
# ------------------------------------------------------------------
# We'll plug in d and names_set after build; see init_word_tools() below.


def init_word_tools(dict_obj, names_iterable, not_words=None):
    """
    Initialize fast lookup layer. Call once after you have `d` and `df_names`.
    
    Parameters
    ----------
    dict_obj : object with .check(str)->bool (e.g., enchant Dict)
    names_iterable : iterable of strings (already lowercase)
    not_words : iterable of lowercase strings to forcibly reject (optional)
    """
    global _DICT_OBJ, _NAMES_SET, _NOT_WORDS
    _DICT_OBJ = dict_obj
    _NAMES_SET = set(names_iterable)
    _NOT_WORDS = set(NOT_WORDS if not_words is None else not_words)
    

    # Clear caches whenever underlying data changes
    is_dictionary_word_cached.cache_clear()
    is_name_cached.cache_clear()
    is_dictionary_word_or_name_cached.cache_clear()


# Default globals (will be replaced by init_word_tools)
_DICT_OBJ = None
_NAMES_SET = set()
_NOT_WORDS = set(NOT_WORDS)
CUSTOM_WORDS = {"sunday","monday",'tuesday','wednesday','thursday','friday','saturday','inlaw','inlaws','backlit','neverland',
                'getcha','meetcha','woah','tonk','tonks','rearview','bitcha','carthy','levittown','cuban','polaroid','orleans',
                'wonderlust','james','memphis','paris','gatlinberg','vegas','noob','noobs','woogie','brazilian','american',
                "rearranger", "truthing","amongst","hauntings","oughta",'momma','icarus','gatlinburg','hemi','cadillac',
                'mona','lisa','hollywood','belinda','satans','jackson','michael','johnson','chelsea','tommy', 'antone', 'wantcha',
                'sophie','tock','aruba','jamaica','amsterdam','cassius','steven','shalaby','usa','napane','parisian','san','bahama',
                'carmen','shit\'ll','december','gemini','stan','april','december','september','gabriel','ireland','spanish',
                'allman','mississippi','drooper','joan','shakey\'s','matthew','brian','cavanaugh','adam','cathy','heathcliff',
                'cleveland','indian','australia','hagar','kristy','bossa','chicago','cinderella','rodina','lolli','joel','marshall',
                'petri','elton','francisco', 'hawaii', 'hawaiian','woulda','coulda','shoulda','marlon','brando','bodi','alice',
                'arkansas','reggie','ella\'s','wayne','watson','jagger','ritalin','inna','nevil','jeaned','coldplay','cara',
                'cleveland\'s','albert','stella','montego','sundays','sunday\'s','monserrat','spain','polaroids','patrick',
                'chopin','donut','pretend\'s','cambodia','alaska','chica','frigerator','heston'
                }

@lru_cache(maxsize=100_000)
def is_dictionary_word_cached(word: str) -> bool:
    """
    Fast dictionary check; expects lowercase.
    Mirrors your original behavior: True for 'a','i'; spell-check for len>1.
    """
    w = word
    if w in {'a', 'i'}:
        return True
    # Check custom words
    if w in CUSTOM_WORDS:
        return True
    # Forced exclusion
    if w in _NOT_WORDS:
        return False
    if len(w) > 1 and _DICT_OBJ is not None:
        # Many dictionary libs are case-sensitive; use lower
        if _DICT_OBJ.check(w):
            return True
    # Step 1: Collapse 3+ repeated characters (e.g., "cooool" -> "cool")
    collapsed = collapse_repeats(word)

    # Step 2: Normalize ii → i and yy → y
    normalized = re.sub("aa", "a", collapsed)
    normalized = re.sub("ii", "i", collapsed)
    normalized = re.sub("yy", "y", normalized)

    # Step 3: Re-check
    if normalized in CUSTOM_WORDS:
        return True
    if normalized in _NOT_WORDS:
        return False
    if len(normalized) > 1 and _DICT_OBJ is not None:
        return _DICT_OBJ.check(normalized)
    
    return False


@lru_cache(maxsize=100_000)
def is_name_cached(word: str) -> bool:
    """Return True if word is in the precomputed names set."""
    return word in _NAMES_SET


@lru_cache(maxsize=100_000)
def is_dictionary_word_or_name_cached(word: str) -> bool:
    """
    Unified validity check with:
      - Forced NOT_WORDS exclusion
      - Dictionary + names
      - Trailing-g variant  (checkin' -> checking approximation)
      - Leading-apostrophe drop ('cause -> cause)
    """
    w = word.lower()

    # Forced exclusion
    if w in _NOT_WORDS:
        return False

    # Main checks
    if is_dictionary_word_cached(w) or is_name_cached(w):
        return True

    # Variants ------------------------------------------------------
    if len(w) > 1:
        # Replace last char w/ 'g' (your original approximate check)
        no_g_word = w[:-1] + 'g'
        if is_dictionary_word_cached(no_g_word) or is_name_cached(no_g_word):
            return True

        # Drop first char (for leading apostrophe cases)
        no_apo_word = w[1:]
        if is_dictionary_word_cached(no_apo_word) or is_name_cached(no_apo_word):
            return True

    return False


# ------------------------------------------------------------------
# 5. JOIN FUNCTION (ALT ORDER): prev3, next3, prev2, next2, prev1, next1
# ------------------------------------------------------------------
def join_non_words_alternate(text):
    words = text.split()
    result = []
    i = 0
    n = len(words)

    #print(f"\nProcessing text: {text}")

    while i < n:
        word = words[i]
        #print(f"[i={i}] Current word: {word}, result so far: {result}")

        # Case: current word valid
        if is_dictionary_word_cached(word.lower()):
            #print(f"  '{word}' is valid -> appending")
            result.append(word)
            i += 1
            continue

        joined = False

        # ----- BACKWARD JOINS -----
        for back in (3, 2, 1):
            if len(result) >= back:
                candidate_tokens = result[-back:] + [word]
                candidate = "".join(t.lower() for t in candidate_tokens)
                #print(f"  Trying backward join {back}: {candidate_tokens} = {candidate}")
                if is_dictionary_word_cached(candidate):
                    merged = "".join(result[-back:] + [word])
                    #rint(f"  **Backward join success: {merged}**")
                    result = result[:-back]
                    result.append(merged)
                    i += 1
                    joined = True
                    break
        if joined:
            continue

        # ----- FORWARD JOINS -----
        for fwd in (3, 2, 1):
            if i + fwd < n:
                candidate_tokens = words[i:i + fwd + 1]
                candidate = "".join(t.lower() for t in candidate_tokens)
                #print(f"  Trying forward join {fwd}: {candidate_tokens} = {candidate}")
                if is_dictionary_word_cached(candidate):
                    merged = "".join(words[i:i + fwd + 1])
                    #print(f"  **Forward join success: {merged}**")
                    result.append(merged)
                    i += fwd + 1
                    joined = True
                    break
        if joined:
            continue

        # ----- No join found -----
        #print(f"  No join found, appending word: {word}")
        result.append(word)
        i += 1

    #print(f"Final result: {' '.join(result)}\n")
    return " ".join(result)





# ------------------------------------------------------------------
# 6. OPTIONAL: APPLY TO A DATAFRAME COLUMN
# ------------------------------------------------------------------
def join_non_words_column(df, text_col, new_col=None, report=False):
    """
    Apply join_non_words_alternate() to every row in a DataFrame column.
    """
    out_col = text_col if new_col is None else new_col
    orig_count = len(df)
    new_texts = []
    changed_rows = 0

    for txt in df[text_col].astype(str):
        new_txt = join_non_words_alternate(txt)
        if new_txt != txt:
            changed_rows += 1
            if report:
                print(f"OLD: {txt}\nNEW: {new_txt}\n---")
        new_texts.append(new_txt)

    df[out_col] = new_texts
    if report:
        print(f"Rows processed: {orig_count}")
        print(f"Rows changed:   {changed_rows}")
    return df


In [ ]:
# Example setup
d = enchant.Dict("en_US")

# Suppose df_names already exists with a 'name' column
names_set = build_names_set(df_names, col="name")

# Extend NOT_WORDS with your exclusions
custom_not_words = {"ly", "ing", "mor", "fol","ya", "mb", "oof","cur","xi","kn","con","soc","en","ting","ling","ding","gain","un",
                    "ta","al","ana","er","ter","writ","fol", "op","uni","mesa","chine","re","val","ley","mo","pa","fore","geth",
                    "es","ted","rd","neath","tween","para","ture","bur","ee","ti","stoop","dow","fro", 'cote','ques','os','era',
                    'hap','lo','ks','fo','fee','wan','ea','rs','str','nge','sago','ht','atomy','fir','incr','cri','mi','swill',
                    'sin','ope','fou','loll','ut','tr','pe','sour','gl','meiny','tome','forme','udo','din', 'saga', 'i\'llhold',
                    'star','ki','farge','deme','forge'
                    }  

# Initialize
init_word_tools(d, names_set, not_words=custom_not_words)


In [49]:
IGNORE_MERGED_WORDS = {"turnaround", "knowhow", "comeback", "allover", "faraway","iii","insomuch","together","halala","walkaway","none","nohow","flyaway","feelgood",
                       "tola", "dada", "baba", "mys", "si", "fro", "tain", "ter", "miked", "noway", "wannabe", "isatin", "rai", "outplayed", "faraway", "inti", "yourside",
                       "alula","there", "goodnight","anytime","fora","aright","outwith","runaway","doit","tome","tod","youthen","begone","tom","seethe","whereto","ami",
                       "somethings","righto","playa","forme","ora", "two", "dado", "caracara", "oho","at","ticktock","av","late","owe","toe","tor","have","udo","abri",
                       "theirs", "could", "tote","aba","dumdum","thee", "ab", "oof", "cause", "dd","dis","meg","hall","then","ire","dingdong", "meal","area","tore",
                       "were","adj","band","hit"
}

def _merge_tokens_concat(tokens):
    """Simple concatenation without overlap handling."""
    return "".join(tokens)


def _merge_tokens_overlap(tokens):
    """
    Concatenate tokens but, at each boundary, if the last char of the
    current merged string == first char of the next token (case-insensitive),
    drop the duplicate boundary char from the next token.
    """
    if not tokens:
        return ""
    merged = tokens[0]
    for t in tokens[1:]:
        if merged and t and merged[-1].lower() == t[0].lower():
            merged += t[1:]
        else:
            merged += t
    return merged


def join_n_words_only(text, n=2, debug=False):
    """
    Attempt to join *exactly n* consecutive whitespace-delimited tokens
    into a single word if (a) the straight concatenation or (b) the
    overlap-trimmed concatenation is a valid dictionary word.

    Parameters
    ----------
    text : str
        Input string.
    n : int >= 2
        Number of consecutive tokens to attempt to merge.
    debug : bool
        If True, print diagnostic info.

    Returns
    -------
    str
        Text with any successful n-word merges applied.
    """
    if n < 2:
        raise ValueError("join_n_words_only: n must be >= 2.")

    words = text.split()
    result = []
    i = 0
    total = len(words)

    if debug:
        print(f"\nProcessing text: {text!r}  (n={n})")

    while i < total:
        # Only attempt merge if at least n tokens remain
        if i + n - 1 < total:
            chunk = words[i:i + n]

            # **NEW CONDITION: For n=2, skip if both are valid words**
            if n == 2:
                w1_valid = is_dictionary_word_cached(chunk[0].lower())
                w2_valid = is_dictionary_word_cached(chunk[1].lower())
                if w1_valid and w2_valid:
                    if debug:
                        print(f"[i={i}] Both words valid -> skip join: {chunk}")
                    result.append(words[i])
                    i += 1
                    continue
                
            # 1. Straight concat
            cand_concat = _merge_tokens_concat(chunk)
            cand_concat_l = cand_concat.lower()

            if debug:
                print(f"[i={i}] Try concat: {chunk} -> {cand_concat_l}")

            if (cand_concat_l not in IGNORE_MERGED_WORDS and
                cand_concat_l not in _NOT_WORDS and
                is_dictionary_word_cached(cand_concat_l)):
                if debug:
                    print(f"  **concat success** -> {cand_concat}")
                result.append(cand_concat)
                i += n
                continue

            # 2. Overlap-aware concat
            cand_overlap = _merge_tokens_overlap(chunk)
            cand_overlap_l = cand_overlap.lower()

            if debug:
                print(f"      Try overlap: {chunk} -> {cand_overlap_l}")

            if (cand_overlap_l not in IGNORE_MERGED_WORDS and
                cand_overlap_l not in _NOT_WORDS and
                is_dictionary_word_cached(cand_overlap_l)):
                if debug:
                    print(f"  **overlap success** -> {cand_overlap}")
                result.append(cand_overlap)
                i += n
                continue

        # No merge: keep current word
        if debug:
            print(f"[i={i}] No merge -> keep {words[i]!r}")
        result.append(words[i])
        i += 1

    out = " ".join(result)
    if debug:
        print(f"Final result: {out!r}\n")
    return out


In [58]:

def split_and_check(word, min_pieces=2, max_pieces=5, is_word_fn=None, debug=False):
    """
    Try to split `word` into between `min_pieces` and `max_pieces` valid pieces.
    Starts at min_pieces (2) and goes up to max_pieces.
    Returns the first successful segmentation found.
    """
    if is_word_fn is None:
        try:
            is_word_fn = is_dictionary_word_cached
        except NameError:
            is_word_fn = is_dictionary_word_cached

    L = len(word)
    if L < min_pieces:
        return False

    # Don't split if the word is already valid
    if is_word_fn(word):
        return False

    @lru_cache(maxsize=None)
    def _is_word_slice(start, end):
        return is_word_fn(word[start:end])

    def _search(start, pieces_left, path):
        remaining_chars = L - start
        if remaining_chars < pieces_left:  # need at least 1 char per piece
            return None

        if pieces_left == 1:
            if _is_word_slice(start, L):
                return path + [word[start:]]
            return None

        max_end = L - (pieces_left - 1)
        for end in range(start + 1, max_end + 1):
            if _is_word_slice(start, end):
                found = _search(end, pieces_left - 1, path + [word[start:end]])
                if found is not None:
                    return found
        return None

    # Start at 2 pieces and go upward
    for num_pieces in range(min_pieces, max_pieces + 1):
        result = _search(0, num_pieces, [])
        if result is not None:
            if debug and num_pieces >= 4:
                print(f"found a {num_pieces}-parter:", word, result)
            return result

    return False

def split_text_and_check(text, is_word_fn=None, min_pieces=2, max_pieces=5, debug=False):
    """
    Apply split_and_check to each token in the string.
    Replace nonwords with a valid segmentation (pieces space-joined) if found.
    """
    if is_word_fn is None:
        try:
            is_word_fn = is_dictionary_word_cached
        except NameError:
            is_word_fn = is_dictionary_word_cached

    words = text.split()
    new_words = []

    BAD_VARIANTS = {"tyg","hg", "dag", "bag","nag","clog","alg","gag","log","sang","stang","fig","pang","frog","shag","wag","sing",
                    'dang','aug','wig','gong'
                    }  
    
    for word in words:
        if is_word_fn(word):
            new_words.append(word)
            continue

        # Variant checks (don't remove internal apostrophes like "i'd")
        found_variant = None
        candidate = word + "g"
        if len(candidate) > 2 and is_word_fn(candidate) and candidate not in BAD_VARIANTS:
            found_variant = candidate

        elif word.endswith("'s"):
            base = word[:-2]  # Remove trailing 's
            if len(base) > 2:
                candidate = base + "g"
                if is_word_fn(candidate) and candidate not in BAD_VARIANTS:
                    found_variant = candidate + "'s"

        elif word.endswith("'"):
            word_no_apos = word.rstrip("'")
            if len(word_no_apos) > 2 and is_word_fn(word_no_apos) and word_no_apos not in BAD_VARIANTS:
                found_variant = word_no_apos
            else:
                candidate = word_no_apos + "g"
                if len(candidate) > 2 and is_word_fn(candidate) and candidate not in BAD_VARIANTS:
                    found_variant = candidate

        if found_variant:
            new_words.append(found_variant)
            continue


        # Try split_and_check
        split_pieces = split_and_check(
            word,
            min_pieces=min_pieces,
            max_pieces=max_pieces,
            is_word_fn=is_word_fn,
            debug=debug,
        )
        if split_pieces:
            new_words.extend(split_pieces)
        else:
            new_words.append(word)

    return " ".join(new_words)


In [51]:
transformed_words_count = 0

def process_transcript(row):
    global transformed_words_count
    text = row['transcript_no_punct']
    orig_text = text
    filename = row['filename'] 

    # Apply transformation steps
    # Order matters here!
    text = collapse_repeats(text)
    text = collapse_known_repeats(text)
    text = normalize_expressive_words(text)
    text = convert_british_to_american(text)
    text = convert_numbers_to_words(text)
    text = split_expressive_string_adjacent(text)
    text = fix_common_splits(text)

    # Join words, starting from the longest joins, and proceeding to the shortest joins 
    for n in range(11, 1, -1):        
        text = join_n_words_only(text, n=n, debug=False)

    text = fix_common_splits(text)

    text = fix_common_splits(text) # might be unnecessary, but shouldn't hurt
    checkpoint_text = text
    text = split_text_and_check(text)
    text = fix_common_splits(text) # might be unnecessary, but shouldn't hurt
    if text != checkpoint_text:
        transformed_words_count += 1
        print("FILE:", filename, "\nOLD:", checkpoint_text, "\nNEW:", text, "\n")
    return text
    text = fix_common_splits(text) # might be unnecessary, but shouldn't hurt

    # Join words, starting from the longest joins, and proceeding to the shortest joins 
    for n in range(11, 1, -1):        
        text = join_n_words_only(text, n=n, debug=False)

    if text != orig_text:
        transformed_words_count += 1
        print("FILE:", filename, "\nOLD:", orig_text, "\nNEW:", text, "\n")
    return text


    text = fix_common_splits(text)

    words = text.split()
    misspelled = [word for word in words if not is_dictionary_word_or_name(word)]
    #if len(misspelled) > 0:
        #print('ORIGINAL:',orig_text,'UPDATED:',words,'MISSPELLED:', misspelled)
    '''corrected = []
    for word in words:
        if word in misspelled:
            print(word,'misspelled!')
            split_test = split_and_check(word)
            if split_test is not False:
                word = split_test
                print(word,'fixed')
                corrected.extend(word)
                continue
            word = correct_misspelling(word)
            print(word,'correction?')
            corrected.append(correct_misspelling(word))
        else:
            corrected.append(word)
    return ' '.join(corrected)'''

# Apply to DataFrame
df['transcript_no_punct_corrected'] = df.apply(process_transcript, axis=1)
print(transformed_words_count, "transformed words!")

FILE: 001940b614eb43f4a0c826d49a67d66d-4.wav 
OLD: butdown inside 
NEW: but down inside 

FILE: 001940b614eb43f4a0c826d49a67d66d-17.wav 
OLD: overand overagain 
NEW: over and over again 

FILE: 001940b614eb43f4a0c826d49a67d66d-21.wav 
OLD: nomeasure of time 
NEW: no measure of time 

FILE: 001940b614eb43f4a0c826d49a67d66d-23.wav 
OLD: thatyou and i 
NEW: that you and i 

FILE: 001940b614eb43f4a0c826d49a67d66d-36.wav 
OLD: overand overagain 
NEW: over and over again 

FILE: 001940b614eb43f4a0c826d49a67d66d-39.wav 
OLD: and i'm talkin' to you 
NEW: and i'm talking to you 

FILE: 001940b614eb43f4a0c826d49a67d66d-40.wav 
OLD: youiknow how you feel 
NEW: you i know how you feel 

FILE: 001940b614eb43f4a0c826d49a67d66d-43.wav 
OLD: overand overagain 
NEW: over and over again 

FILE: 007c0152242340008ff45781a9b08546-0.wav 
OLD: we were goin' way too fast 
NEW: we were going way too fast 

FILE: 007c0152242340008ff45781a9b08546-1.wav 
OLD: chasin' down the hourglass 
NEW: chasing down the hour

In [52]:
len(df)

153681

In [53]:
# Instantiate the tokenizer
model_name = "openai/whisper-base"
language = "english" # Change to your dataset's language
task = "transcribe" # Use "translate" if you're translating to English

processor = WhisperProcessor.from_pretrained(model_name, language=language, task=task)
tokenizer = processor.tokenizer

c:\Users\Jared\anaconda3\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [54]:
# Checks if something is a word based on corpus frequency
def is_known_word(word, threshold=5.5):
    freq = zipf_frequency(word, 'en')
    return freq >= threshold, freq

is_known_word('hello')

(False, 4.72)

In [55]:
# Test how a word is tokenized here
word = 'whoo'
tokens = tokenizer.tokenize(word)
print(tokens)

['who', 'o']


In [56]:
# Test if things are words here
word = "darling"
print(is_dictionary_word(word))

name = 'francisco'
print(is_name(name))

True
True


In [57]:
# All of this is only needed if dealing with words, not lines

# Correctly format all lines (including nan formatting)
#def parse_and_check_for_nan(val):
#    if isinstance(val, str):
#        nan_count = len(re.findall(r"'word': nan", val))
#        if nan_count > 0:
#            print(f"'word': nan appears {nan_count} times")
#
#        val_fixed = re.sub(r"'word': nan", "'word': np.nan", val)
#        try:
#            parsed = eval(val_fixed, {"np": np})
#            if any(pd.isna(item.get('word')) for item in parsed):
#               return None
#            return parsed
#        except Exception:
#            return None
#    return None

# Format all lines and remove those containing nan
#df['words'] = df['words'].apply(parse_and_check_for_nan)
#df = df.dropna(subset=['words']).reset_index(drop=True)

# Clean 'words' by deleting duplicate timestamps
#def remove_duplicate_dicts(lst):
#    seen = set()
#    result = []
#    for d in lst:
#        key = tuple(sorted(d.items()))
#        if key not in seen:
#            seen.add(key)
#            result.append(d)
#    return result

# Clean 'words' by deleting timestamps which end before they start
#def remove_invalid_time_dicts(lst):
#    return [d for d in lst if d.get('end', 0) >= d.get('start', 0)]

# Get rid of duplicate and invalid timestamps
#df['words'] = df['words'].apply(lambda lst: remove_invalid_time_dicts(remove_duplicate_dicts(lst)) if isinstance(lst, list) else lst)

# Create column without timestamps
#df['words_no_time'] = df['words'].apply(lambda lst: " ".join(item['word'] for item in lst) if isinstance(lst, list) else "")

# Used to further align transcript with words
#def extract_aligned_words(row):
#    transcript_words = row['transcript_no_punct'].split()
#    words_no_time = row['words_no_time'].replace(" ", "").replace("'", "")
#
#    aligned_words = []
#    for word in transcript_words:
#        clean_word = word.replace("'", "")
#        pointer = 0
#        for char in clean_word:
#            pointer = words_no_time.find(char, pointer)
#            if pointer == -1:
#                break
#            pointer += 1
#        else:
#            aligned_words.append(word)
#
#    return " ".join(aligned_words)
#
#df['transcript_words_align'] = df.apply(extract_aligned_words, axis=1)

# Used to further align transcript with words
#def intersect_words_with_transcript_align(row):
#    transcript_words = row['transcript_words_align'].split()
#    words_no_time_clean = row['words_no_time'].replace(" ", "").replace("'", "")
    
#    result_words = []
#    pointer = 0

#    for word in transcript_words:
#        clean_word = word.replace("'", "")
#        temp_pointer = pointer  # Start checking from current position

#        for char in clean_word:
#            temp_pointer = words_no_time_clean.find(char, temp_pointer)
#            if temp_pointer == -1:
#                break
#            temp_pointer += 1
#        else:
#            # If the entire word matched, update the main pointer and keep the word
#            pointer = temp_pointer
#            result_words.append(word)

#    return " ".join(result_words)


#df['transcript_words_intersection'] = df.apply(intersect_words_with_transcript_align, axis=1)



#def restore_from_reference_row(row):
#    compressed = row['words_no_time']
#    reference = row['transcript_no_punct']

#    compressed_clean = compressed.replace(" ", "").replace("'", "")
#    reference_clean = reference.replace(" ", "").replace("'", "")

#    start_index = reference_clean.find(compressed_clean)
#    if start_index == -1:
#        return ""

#    result = []
#    ref_char_pos = 0
#    matched_chars = 0
    
#    for char in reference:
#        if char not in {" ", "'"}:
#            if ref_char_pos >= start_index and matched_chars < len(compressed_clean):
#                result.append(char)
#                matched_chars += 1
#            elif matched_chars > 0 and matched_chars < len(compressed_clean):
#                result.append(char)
#            ref_char_pos += 1
#        elif matched_chars > 0 and matched_chars < len(compressed_clean):
#            result.append(char)

#        if matched_chars == len(compressed_clean):
#            break

#    return ''.join(result).strip()

#df['transcript_words_restored'] = df.apply(restore_from_reference_row, axis=1)






#pd.set_option('display.max_colwidth', None)
#pd.set_option("display.max_rows", None) 
#pd.set_option("display.width", 0)  
#def fix_transcript_list(transcript):
#    transcript_list = transcript.split(" ")
#    split_transcript_list = []
#    
#    for word in transcript_list:
#        if not is_dictionary_word(word):
#            if not is_name(word):
#                split_attempt = split_and_check(word)
#                if split_attempt is False:
#                    split_transcript_list.append(word)
#                else:
#                    split_transcript_list.extend(split_attempt)
#            else:
#                split_transcript_list.append(word)
#        else:
#            split_transcript_list.append(word)
#    
#    return ' '.join(split_transcript_list)

#df.loc[df.index[-200:], 'transcript_words_restored_fixed'] = df.loc[df.index[-200:], 'transcript_words_restored'].apply(fix_transcript_list)
#df.loc[df.index[-200:],['transcript_no_punct','words_no_time','transcript_words_restored_fixed','filename']]







#def check_non_whitespace_apostrophe_match(row):
#    a = re.sub(r"[ '\t\n\r\f\v]", "", row['words_no_time'])
#    b = re.sub(r"[ '\t\n\r\f\v]", "", row['transcript_words_restored'])

#    return a != b

# Apply across the DataFrame
#disagreements = df.apply(check_non_whitespace_apostrophe_match, axis=1)
#disagreement_count = disagreements.sum()
#mismatches = df[disagreements]

#print(f"Number of disagreeing rows (ignoring whitespace/apostrophes): {disagreement_count}")



# This is not currently needed
# Used to look at differences exist between lines and words
#df[['filename','transcript_no_punct','words_no_time','transcript_words_restored']].iloc[29763]
#pd.set_option("display.max_rows", None) 
#pd.set_option("display.max_columns", None)
#pd.set_option("display.width", 0)  # Automatically fit to content width
#pd.set_option("display.max_colwidth", None)
#pd.set_option("display.expand_frame_repr", False)  # Disable line wrapping for wide frames
#mismatches = df[disagreements]
#print(mismatches[['filename','transcript_no_punct','words_no_time']])